In [163]:
YEAR = 2024

# AIRPORT = "JFK"
# AIRPORT = "ORD"
AIRPORT = "ATL"

PROCESSED_DATA_FILE = f"../data/aspm/processed/{AIRPORT}_{YEAR}.csv"
CLEAN_DATA_FILE = f"../data/aspm/cleaned/{AIRPORT}_{YEAR}.csv"

In [164]:
import pandas as pd

In [165]:
df = pd.read_csv(PROCESSED_DATA_FILE)

***
# Validate the processed ASPM data

These checks look for date, hour, duplicate, coverage, missing-value, numeric, airport-code, and row-order problems. They create temporary validation objects but do not change, sort, filter, or save `df`.
***

***
## Check the report dates

Parse the dates into a temporary Series. This confirms that the values can become datetimes without replacing the original `report_date` column.
***

In [166]:
parsed_report_date = pd.to_datetime(df["report_date"], errors="coerce")

date_validation = pd.Series({
    "source dtype": str(df["report_date"].dtype),
    "parsed dtype": str(parsed_report_date.dtype),
    "unparseable or missing dates": int(parsed_report_date.isna().sum()),
    "dates outside selected year": int(
        (parsed_report_date.notna() & (parsed_report_date.dt.year != YEAR)).sum()
    ),
    "first date": parsed_report_date.min(),
    "last date": parsed_report_date.max()
}, name="report_date validation")

date_validation

source dtype                                 object
parsed dtype                         datetime64[ns]
unparseable or missing dates                      0
dates outside selected year                       0
first date                      2024-01-01 00:00:00
last date                       2024-12-31 00:00:00
Name: report_date validation, dtype: object

***
## Check hours, duplicate keys, and hourly coverage

First determine whether `Hour` follows a 0–23 or 1–24 convention. Then check that each airport, date, and hour appears once and that there are no gaps in the hourly timeline. Temporary timestamps are used only for these checks.
***

In [167]:
hour_values = pd.to_numeric(df["Hour"], errors="coerce")
uses_zero_to_23 = bool(hour_values.notna().all() and hour_values.between(0, 23).all())
uses_one_to_24 = bool(hour_values.notna().all() and hour_values.between(1, 24).all())

hour_validation = pd.Series({
    "source dtype": str(df["Hour"].dtype),
    "nonnumeric or missing hours": int(hour_values.isna().sum()),
    "minimum hour": hour_values.min(),
    "maximum hour": hour_values.max(),
    "all values fit 0-23": uses_zero_to_23,
    "all values fit 1-24": uses_one_to_24
}, name="Hour validation")

hour_validation

source dtype                   int64
nonnumeric or missing hours        0
minimum hour                       0
maximum hour                      23
all values fit 0-23             True
all values fit 1-24            False
Name: Hour validation, dtype: object

***
### Check for duplicate records

Count both fully repeated rows and repeated airport-date-hour keys. Each airport should normally have only one record for each date and hour.
***

In [168]:
duplicate_validation = pd.Series({
    "fully duplicated rows": int(df.duplicated().sum()),
    "duplicate airport-date-hour keys": int(
        df.duplicated(subset=["airport", "report_date", "Hour"], keep=False).sum()
    )
}, name="duplicate validation")

duplicate_validation

fully duplicated rows               0
duplicate airport-date-hour keys    0
Name: duplicate validation, dtype: int64

***
### Check for missing hours

Build a temporary timestamp from the parsed date and hour. Compare the observed timestamps with a complete hourly range for each airport, then show up to 20 missing-hour examples. This does not add the timestamp to `df`.
***

In [169]:
if uses_zero_to_23:
    hour_offsets = hour_values
    detected_hour_convention = "0-23"
elif uses_one_to_24:
    hour_offsets = hour_values - 1
    detected_hour_convention = "1-24"
else:
    hour_offsets = pd.Series(float("nan"), index=df.index)
    detected_hour_convention = "invalid or mixed"

validation_timestamp = (
    parsed_report_date + pd.to_timedelta(hour_offsets, unit="h")
)

hourly_keys = pd.DataFrame({
    "airport": df["airport"],
    "timestamp": validation_timestamp
}).dropna().drop_duplicates()

coverage_rows = []
missing_hour_rows = []

for airport, airport_hours in hourly_keys.groupby("airport"):
    observed = pd.DatetimeIndex(airport_hours["timestamp"].sort_values())
    expected = pd.date_range(observed.min(), observed.max(), freq="h")
    missing = expected.difference(observed)

    coverage_rows.append({
        "airport": airport,
        "first timestamp": observed.min(),
        "last timestamp": observed.max(),
        "observed hours": len(observed),
        "expected hours": len(expected),
        "missing hours": len(missing)
    })

    missing_hour_rows.extend(
        {"airport": airport, "missing timestamp": timestamp}
        for timestamp in missing
    )

coverage_summary = pd.DataFrame(coverage_rows)
missing_hour_examples = pd.DataFrame(missing_hour_rows)

detected_hour_convention, coverage_summary, missing_hour_examples.head(20)

('0-23',
   airport first timestamp      last timestamp  observed hours  expected hours  \
 0     ATL      2024-01-01 2024-12-31 23:00:00            8769            8784   
 
    missing hours  
 0             15  ,
    airport   missing timestamp
 0      ATL 2024-01-02 03:00:00
 1      ATL 2024-02-07 01:00:00
 2      ATL 2024-02-08 01:00:00
 3      ATL 2024-03-17 03:00:00
 4      ATL 2024-03-31 03:00:00
 5      ATL 2024-04-05 03:00:00
 6      ATL 2024-04-13 03:00:00
 7      ATL 2024-04-30 02:00:00
 8      ATL 2024-07-05 03:00:00
 9      ATL 2024-08-25 02:00:00
 10     ATL 2024-10-30 02:00:00
 11     ATL 2024-12-03 02:00:00
 12     ATL 2024-12-17 02:00:00
 13     ATL 2024-12-25 02:00:00
 14     ATL 2024-12-25 03:00:00)

***
## Check missing and numeric values

Count missing values, values that cannot be read as numbers, infinite values, and negative scheduled-flight counts. The summary statistics help identify values that need investigation. Negative delay averages may be valid and are not removed here.
***

In [170]:
missing_validation = pd.DataFrame({
    "missing count": df.isna().sum(),
    "missing percent": df.isna().mean().mul(100)
})

missing_validation

,missing count,missing percent
airport,0,0.0
report_date,0,0.0
Hour,0,0.0
Scheduled Departures,0,0.0
Scheduled Arrivals,0,0.0
Average Taxi Out Time,0,0.0
Average Taxi Out Delay,0,0.0
Average Airborne Delay,0,0.0
Average Taxi In Delay,0,0.0


***
### Check numeric columns

Try converting the expected numeric fields in a temporary dataframe. Report values that fail conversion, infinite values, and negative scheduled arrival or departure counts.
***

In [171]:
expected_numeric_columns = [
    "Hour",
    "Scheduled Departures",
    "Scheduled Arrivals",
    "Average Taxi Out Time",
    "Average Taxi Out Delay",
    "Average Airborne Delay",
    "Average Taxi In Delay"
]
numeric_columns = [column for column in expected_numeric_columns if column in df.columns]
numeric_values = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

numeric_validation = pd.DataFrame({
    "source dtype": df[numeric_columns].dtypes.astype(str),
    "failed numeric conversion": [
        int((df[column].notna() & numeric_values[column].isna()).sum())
        for column in numeric_columns
    ],
    "infinite values": [
        int(numeric_values[column].isin([float("inf"), float("-inf")]).sum())
        for column in numeric_columns
    ]
}, index=numeric_columns)

scheduled_columns = ["Scheduled Departures", "Scheduled Arrivals"]
negative_scheduled_counts = (numeric_values[scheduled_columns] < 0).sum()

numeric_validation, negative_scheduled_counts

(                       source dtype  failed numeric conversion  \
 Hour                          int64                          0   
 Scheduled Departures          int64                          0   
 Scheduled Arrivals            int64                          0   
 Average Taxi Out Time       float64                          0   
 Average Taxi Out Delay      float64                          0   
 Average Airborne Delay      float64                          0   
 Average Taxi In Delay       float64                          0   
 
                         infinite values  
 Hour                                  0  
 Scheduled Departures                  0  
 Scheduled Arrivals                    0  
 Average Taxi Out Time                 0  
 Average Taxi Out Delay                0  
 Average Airborne Delay                0  
 Average Taxi In Delay                 0  ,
 Scheduled Departures    0
 Scheduled Arrivals      0
 dtype: int64)

***
### Review numeric ranges

Display summary statistics for each numeric field. Very large, very small, or unexpected values should be investigated before feature engineering, but this step does not remove or replace them.
***

In [172]:
numeric_values.describe().T

,count,mean,std,min,25%,50%,75%,max
Hour,8769.0,11.515680,6.918040,0.0,6.00,12.00,18.00,23.00
Scheduled Departures,8769.0,43.764055,30.591524,0.0,7.00,51.00,69.00,113.00
Scheduled Arrivals,8769.0,43.686851,29.399134,0.0,13.00,54.00,68.00,117.00
Average Taxi Out Time,8769.0,14.902892,5.996419,0.0,14.00,15.45,17.00,123.00
Average Taxi Out Delay,8769.0,3.640293,3.790007,0.0,1.92,3.04,4.42,110.90
Average Airborne Delay,8769.0,1.580325,3.461422,0.0,0.48,0.92,1.64,114.65
Average Taxi In Delay,8769.0,3.068796,2.799415,0.0,1.40,2.53,4.00,56.60


***
## Check airport codes and row order

Check for missing, unexpected, lowercase, or padded airport codes. Then confirm that the current rows are already ordered by airport and hourly timestamp. These temporary checks do not update the airport values or sort `df`.
***

In [173]:
airport_text = df["airport"].astype("string")
normalized_airport = airport_text.str.strip().str.upper()

airport_validation = pd.Series({
    "missing airport codes": int(df["airport"].isna().sum()),
    "unique normalized airport codes": int(normalized_airport.nunique(dropna=True)),
    "codes needing trim or uppercase": int(
        (airport_text.notna() & airport_text.ne(normalized_airport)).sum()
    ),
    f"codes other than {AIRPORT}": int(
        (normalized_airport.notna() & normalized_airport.ne(AIRPORT)).sum()
    )
}, name="airport validation")

airport_validation, normalized_airport.value_counts(dropna=False)

(missing airport codes              0
 unique normalized airport codes    1
 codes needing trim or uppercase    0
 codes other than ATL               0
 Name: airport validation, dtype: int64,
 airport
 ATL    8769
 Name: count, dtype: Int64)

***
### Check the current row order

Compare the existing row order with airport and timestamp order. This reports whether sorting is needed without changing the dataframe.
***

In [174]:
order_keys = pd.DataFrame({
    "airport": normalized_airport,
    "timestamp": validation_timestamp
})
expected_order = order_keys.sort_values(
    ["airport", "timestamp"],
    kind="stable",
    na_position="last"
).index

pd.Series({
    "rows already in chronological airport order": bool(expected_order.equals(df.index)),
    "timestamps missing because date/hour is invalid": int(validation_timestamp.isna().sum())
}, name="row-order validation")

rows already in chronological airport order        True
timestamps missing because date/hour is invalid       0
Name: row-order validation, dtype: object

***
## Sort the dataframe

Sort the rows by the normalized airport code and temporary hourly timestamp, then reset the index. This changes only row order; it does not change any stored values, add columns, remove rows, or write a file.
***

In [175]:
sort_order = order_keys.sort_values(
    ["airport", "timestamp"],
    kind="stable",
    na_position="last"
).index

df = df.loc[sort_order].reset_index(drop=True)
df[["airport", "report_date", "Hour"]].head()

,airport,report_date,Hour
0,ATL,2024-01-01,0
1,ATL,2024-01-01,1
2,ATL,2024-01-01,2
3,ATL,2024-01-01,3
4,ATL,2024-01-01,4


***
## Convert the report date

The validation found no missing or unparseable dates, so convert `report_date` from CSV text to pandas `datetime64[ns]` values.
***

In [176]:
df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")

***
## Create the hourly DATE field

Combine `report_date` with `Hour` to create one timestamp for each ASPM record. Minutes are set to zero because ASPM observations describe complete clock hours. This `DATE` field will be used to join ASPM data to the matching BTS hour.
***

In [177]:
df["DATE"] = (
    df["report_date"]
    + pd.to_timedelta(df["Hour"], unit="h")
)

***
## Save the cleaned and validated ASPM file

Create the output folder if it does not exist, then save the sorted dataframe without the pandas index.
***

In [178]:
from pathlib import Path

Path(CLEAN_DATA_FILE).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_DATA_FILE, index=False)

In [179]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8769 entries, 0 to 8768
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   airport                 8769 non-null   object        
 1   report_date             8769 non-null   datetime64[ns]
 2   Hour                    8769 non-null   int64         
 3   Scheduled Departures    8769 non-null   int64         
 4   Scheduled Arrivals      8769 non-null   int64         
 5   Average Taxi Out Time   8769 non-null   float64       
 6   Average Taxi Out Delay  8769 non-null   float64       
 7   Average Airborne Delay  8769 non-null   float64       
 8   Average Taxi In Delay   8769 non-null   float64       
 9   DATE                    8769 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(4), int64(3), object(1)
memory usage: 685.2+ KB


In [180]:
df.head()

,airport,report_date,Hour,Scheduled Departures,Scheduled Arrivals,Average Taxi Out Time,Average Taxi Out Delay,Average Airborne Delay,Average Taxi In Delay,DATE
0,ATL,2024-01-01,0,0,9,0.0,0.0,0.43,1.46,2024-01-01 00:00:00
1,ATL,2024-01-01,1,0,0,0.0,0.0,0.00,0.00,2024-01-01 01:00:00
2,ATL,2024-01-01,2,0,0,0.0,0.0,3.00,0.00,2024-01-01 02:00:00
3,ATL,2024-01-01,3,0,0,0.0,0.0,0.00,0.00,2024-01-01 03:00:00
4,ATL,2024-01-01,4,0,1,0.0,0.0,0.00,14.60,2024-01-01 04:00:00
